In [1]:
import boto3
import os
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
from io import BytesIO

from ilipy import Session
from ilipy.database import DistanceCorrelation
from ilipy import TrackIndex, OdometerTicks

inspection_id = "0ABP0TFUSH1"
num_tracks = 22
environment = "prod"
model_detection_threshold = 0.99

id_surface_fld = f"s3://dv-ilit0008/track_runs/{inspection_id}/ili_ml_surface/v1.2/"

In [ ]:
def download_parquet_from_s3(bucket_name, s3_key, local_path=None):
    """
    Download a parquet file from S3 and optionally save locally or return as DataFrame
    
    Args:
        bucket_name: S3 bucket name
        s3_key: S3 key (path) for the file
        local_path: Optional local file path to save the file. If None, returns DataFrame
        
    Returns:
        DataFrame if local_path is None, otherwise saves file locally and returns file path
    """
    s3_client = boto3.client('s3')
    
    try:
        # Download file from S3
        response = s3_client.get_object(Bucket=bucket_name, Key=s3_key)
        
        if local_path:
            # Save to local file
            with open(local_path, 'wb') as f:
                f.write(response['Body'].read())
            print(f"File downloaded and saved to: {local_path}")
            return local_path
        else:
            # Return as DataFrame
            parquet_buffer = BytesIO(response['Body'].read())
            df = pd.read_parquet(parquet_buffer)
            print(f"Downloaded DataFrame with {len(df)} rows from s3://{bucket_name}/{s3_key}")
            return df
            
    except Exception as e:
        print(f"Error downloading file: {e}")
        return None


In [ ]:
def download_s3_folder(bucket_name: str, s3_folder: str, local_dir: str = None):
    """
    Download the contents of a folder directory
    Args:
        bucket_name: the name of the s3 bucket
        s3_folder: the folder path in the s3 bucket
        local_dir: the dir data is downloaded to
    """
    s3 = boto3.resource("s3")
    bucket = s3.Bucket(bucket_name)
    for obj in bucket.objects.filter(Prefix=s3_folder):
        target = (
            obj.key
            if local_dir is None
            else os.path.join(local_dir, os.path.relpath(obj.key, s3_folder))
        )
        if not os.path.exists(os.path.dirname(target)):
            os.makedirs(os.path.dirname(target))
        if obj.key[-1] == "/":
            continue
        if not os.path.exists(target):
            bucket.download_file(obj.key, target)
            print("Downloading: {}".format(obj.key))

bucket_name = "dv-ilit0010"   

s3_key = f"track_runs/{inspection_id}/ili_ml_surface/"
#download_s3_folder(bucket_name, s3_key, f"./data-files/insp_{inspection_id}_ml_surface")

In [ ]:
import numpy as np

def ransac_line_fit(x, y, n_iters=500, distance_threshold=0.005, min_inlier_ratio=0.5, random_state=None):
    """
    Fit a robust line y = m x + b to 1D data (x, y) using RANSAC.

    Args:
        x, y: 1D arrays of the same length.
        n_iters: number of RANSAC iterations.
        distance_threshold: max perpendicular distance (in y units) to count a point as an inlier.
        min_inlier_ratio: early-stop if a model has this fraction of inliers.
        random_state: optional int for reproducibility.

    Returns:
        m, b, inlier_mask
        where:
          y ≈ m * x + b
          inlier_mask is a boolean array of same length as x/y.
    """
    x = np.asarray(x, dtype=float)
    y = np.asarray(y, dtype=float)
    assert x.shape == y.shape
    n_points = x.size

    rng = np.random.default_rng(random_state)

    best_m, best_b = None, None
    best_inlier_mask = None
    best_inlier_count = 0

    # Helper: fit line from two points
    def line_from_points(x1, y1, x2, y2):
        if x2 == x1:
            # vertical line -> we handle this by skipping that sample
            return None, None
        m = (y2 - y1) / (x2 - x1)
        b = y1 - m * x1
        return m, b

    for _ in range(n_iters):
        # 1) Sample two distinct points
        idx = rng.choice(n_points, size=2, replace=False)
        x1, y1 = x[idx[0]], y[idx[0]]
        x2, y2 = x[idx[1]], y[idx[1]]

        m, b = line_from_points(x1, y1, x2, y2)
        if m is None:
            continue  # skip vertical-line degeneracy

        # 2) Compute perpendicular distance of all points to this line
        # Line: y = m x + b
        # Distance in y direction is |y - (m x + b)|
        # (for small slopes this is fine; for general case we can normalize)
        y_pred = m * x + b
        residuals = np.abs(y - y_pred)

        inlier_mask = residuals < distance_threshold
        inlier_count = np.count_nonzero(inlier_mask)

        # 3) Keep best model
        if inlier_count > best_inlier_count:
            best_inlier_count = inlier_count
            best_m, best_b = m, b
            best_inlier_mask = inlier_mask

            # Early stop if we already explain most points
            if inlier_count >= min_inlier_ratio * n_points:
                break

    if best_inlier_mask is None:
        raise RuntimeError("RANSAC failed to find a valid model")

    # 4) Refit line using only inliers (ordinary least squares)
    x_in = x[best_inlier_mask]
    y_in = y[best_inlier_mask]

    # [m, b] = argmin ||y - (m x + b)||
    A = np.column_stack([x_in, np.ones_like(x_in)])
    m_refined, b_refined = np.linalg.lstsq(A, y_in, rcond=None)[0]

    return m_refined, b_refined, best_inlier_mask

def add_curvature_columns(df):
    """
    For each row in df, take 801 columns (df.iloc[:, 4:-1]), 
    compute global circle curvature and RANSAC line fit metrics.
    """
    feature_cols = df.columns[4:-1]
    n_points = len(feature_cols)
    x = np.arange(n_points)
    
    # Extract all y values as a 2D numpy array (rows x points)
    y_matrix = df[feature_cols].values.astype(float)
    
    # Initialize result lists
    ransac_slopes = []
    ransac_intercepts = []
    ransac_inlier_counts = []
    ransac_inlier_ratios = []
    
    print(f"Processing {len(df)} rows with {n_points} points each...")
    
    for i in range(len(df)):
        y = y_matrix[i]
    
        
        # RANSAC line fit
        try:
            m, b, inlier_mask = ransac_line_fit(
                x, y,
                n_iters=500,
                distance_threshold=0.005,
                min_inlier_ratio=0.7,
                random_state=0
            )
            inlier_count = np.count_nonzero(inlier_mask)
            inlier_ratio = inlier_count / n_points
            
            ransac_slopes.append(m)
            ransac_intercepts.append(b)
            ransac_inlier_counts.append(inlier_count)
            ransac_inlier_ratios.append(inlier_ratio)
        except Exception as e:
            ransac_slopes.append(np.nan)
            ransac_intercepts.append(np.nan)
            ransac_inlier_counts.append(0)
            ransac_inlier_ratios.append(0.0)
            
        if (i + 1) % 100 == 0:
            print(f"  Processed {i + 1}/{len(df)} rows")
    
    # Create result dataframe
    result_df = df.drop(columns=feature_cols).copy()

    
    # Add RANSAC line fit columns
    result_df['ransac_slope'] = ransac_slopes
    result_df['ransac_intercept'] = ransac_intercepts
    result_df['ransac_inlier_count'] = ransac_inlier_counts
    result_df['ransac_inlier_ratio'] = ransac_inlier_ratios
    

    return result_df

In [ ]:
def add_view_distances(df, dist_corr, track_col='track_id', start_odo_col='start_odometer_tick', end_odo_col='end_odometer_tick'):
    """
    Add view_distance_start and view_distance_stop columns to DataFrame
    
    Args:
        df: DataFrame with odometer tick columns
        dist_corr: DistanceCorrelation object from ilipy
        track_col: column name containing track IDs
        start_odo_col: column name containing start odometer ticks
        end_odo_col: column name containing end odometer ticks
    
    Returns:
        DataFrame with added view_distance_start and view_distance_stop columns
    """
    if df is None or len(df) == 0:
        return df
    
    df = df.copy()
    
    # Initialize new columns
    df['view_distance_start'] = np.nan
    df['view_distance_stop'] = np.nan
    
    # Process each row
    for idx, row in df.iterrows():
        try:
            track_id = int(row[track_col])
            start_odo = int(row[start_odo_col])
            end_odo = int(row[end_odo_col])
            
            # Get view distances
            start_vd = dist_corr.get_view_distance_from_odometer_ticks(
                TrackIndex(track_id), 
                OdometerTicks(start_odo)
            ).value
            
            end_vd = dist_corr.get_view_distance_from_odometer_ticks(
                TrackIndex(track_id), 
                OdometerTicks(end_odo)
            ).value
            df.loc[idx, 'view_distance_start'] = start_vd
            df.loc[idx, 'view_distance_stop'] = end_vd
            
        except Exception as e:
            print(f"Error processing row {idx}: {e}")
            continue
    return df

In [ ]:
session = Session(environment=environment)
session.set_active_inspection(inspection_id)
dist_corr = DistanceCorrelation(session)

bucket_name = "dent-dev"
num_tracks = 22
bookmarks_dfs = []
for track_idx in range(num_tracks):
    s3_key = f"dent_bookmarks/UltrasoundDent_bookmarks_{inspection_id}_track_{track_idx}.parquet"
    df = download_parquet_from_s3(bucket_name, s3_key)
    if df is not None:
        print(f"Downloaded DataFrame shape: {df.shape}")
        processed_df = add_view_distances(df, dist_corr)
        bookmarks_dfs.append(processed_df)
bookmarks_dfs = pd.concat(bookmarks_dfs, ignore_index=True)
display(bookmarks_dfs)

In [ ]:
bookmarks_dfs = bookmarks_dfs[bookmarks_dfs['start_odometer_tick'] != bookmarks_dfs['end_odometer_tick']]
display(bookmarks_dfs)

In [2]:
#save bookmarks_dfs
#bookmarks_dfs.to_parquet(f"./insp_{inspection_id}_bookmarks_with_view_distances.parquet", index=False)
bookmarks_dfs = pd.read_parquet(f"./insp_{inspection_id}_bookmarks_with_view_distances.parquet")


In [ ]:
# import glob
# import re
# from collections import defaultdict

# def merge_surface_curvature_to_dents(dent_df, surface_base_path):
#     """
#     Merge surface curvature data into dent dataframe based on track_id and frame ranges.
#     Optimized version with batch processing and reduced file I/O.
#     Filters surface data before computing curvature calculations.
#     Computes RANSAC line fit summary statistics directly without storing lists.
    
#     Args:
#         dent_df: DataFrame with dent bookmarks containing track_id, start_frame, end_frame
#         surface_base_path: Base path to surface data files
    
#     Returns:
#         DataFrame with added summary statistics (no lists stored)
#     """
#     if dent_df is None or len(dent_df) == 0:
#         return dent_df
    
#     dent_df = dent_df.copy()
    
#     # Initialize summary statistics columns only
#     dent_df['ransac_inlier_ratio_mean'] = np.nan
#     dent_df['ransac_inlier_ratio_median'] = np.nan
#     dent_df['ransac_inlier_ratio_min'] = np.nan



#     # Group dents by track_id for batch processing
#     dents_by_track = defaultdict(list)
#     for idx, row in dent_df.iterrows():
#         dents_by_track[row['track_id']].append({
#             'idx': idx,
#             'start_frame': row['start_frame'],
#             'end_frame': row['end_frame']
#         })
    
#     # Process each track
#     for track_id, dent_list in dents_by_track.items():
#         track_str = f"track_{track_id:02d}"
#         track_path = f"{surface_base_path}/v1.2/{track_str}/**/frames_*.parquet"
        
#         # Find all parquet files for this track
#         parquet_files = glob.glob(track_path, recursive=True)
        
#         # Parse file metadata once
#         file_metadata = []
#         for parquet_file in parquet_files:
#             match = re.search(r'frames_(\d+)_(\d+)_\d+\.parquet', parquet_file)
#             if match:
#                 file_metadata.append({
#                     'path': parquet_file,
#                     'start': int(match.group(1)),
#                     'end': int(match.group(2))
#                 })
        
#         # Sort files by start frame for efficient searching
#         file_metadata.sort(key=lambda x: x['start'])
        
#         # For each file, find all dents that overlap with it
#         for file_info in file_metadata:
#             file_start = file_info['start']
#             file_end = file_info['end']
            
#             # Find overlapping dents
#             overlapping_dents = [
#                 d for d in dent_list
#                 if d['start_frame'] < file_end and d['end_frame'] >= file_start
#             ]
            
#             if not overlapping_dents:
#                 continue
            
#             # Load file once for all overlapping dents
#             try:
#                 print(f"Loading: {file_info['path']} for {len(overlapping_dents)} dents")
#                 surface_df = pd.read_parquet(file_info['path'])
                
#                 # Filter by track_id first
#                 surface_df = surface_df[surface_df['track_id'] == track_id]
                
#                 # Get frame range for all overlapping dents
#                 min_frame = min(d['start_frame'] for d in overlapping_dents)
#                 max_frame = max(d['end_frame'] for d in overlapping_dents)
                
#                 # Filter surface data to only frames needed by dents BEFORE curvature computation
#                 surface_df = surface_df[
#                     (surface_df['frame'] >= min_frame) & 
#                     (surface_df['frame'] <= max_frame)
#                 ]
                
#                 if len(surface_df) == 0:
#                     print(f"  No matching frames in range [{min_frame}, {max_frame}]")
#                     continue
                
#                 print(f"  Computing RANSAC metrics for {len(surface_df)} filtered frames...")
#                 # Now compute RANSAC metrics only on filtered data
#                 surface_df = add_curvature_columns(surface_df)
                
#                 # Process each overlapping dent
#                 for dent in overlapping_dents:
#                     mask = (
#                         (surface_df['frame'] >= dent['start_frame']) & 
#                         (surface_df['frame'] <= dent['end_frame'])
#                     )
#                     matching_surface = surface_df[mask]
                    
#                     if len(matching_surface) > 0:
#                         # Compute statistics directly from matching data
#                         inlier_ratios = matching_surface['ransac_inlier_ratio'].values
#                         slopes = matching_surface['ransac_slope'].values
                        
#                         # Remove NaN values
#                         valid_ratios = inlier_ratios[~np.isnan(inlier_ratios)]
#                         valid_slopes = slopes[~np.isnan(slopes)]
                        
#                         if len(valid_ratios) > 0:
#                             dent_df.at[dent['idx'], 'ransac_inlier_ratio_mean'] = np.mean(valid_ratios)
#                             dent_df.at[dent['idx'], 'ransac_inlier_ratio_median'] = np.median(valid_ratios)
#                             dent_df.at[dent['idx'], 'ransac_inlier_ratio_min'] = np.min(valid_ratios)
                        
                        
#             except Exception as e:
#                 print(f"Error loading {file_info['path']}: {e}")
#                 continue
    
#     print(f"\nMerged curvature data for {len(dent_df)} dent records")
#     print(f"Added RANSAC line fit summary statistics")
    
#     return dent_df

# # Usage:
# surface_base_path = f"./data-files/insp_{inspection_id}_ml_surface"
# merged_dents = merge_surface_curvature_to_dents(bookmarks_dfs, surface_base_path)

# # Display key metrics
# display(merged_dents)


In [ ]:
#merged_dents.to_parquet(f"./data-files/insp_{inspection_id}_refined_dents_with_surface.parquet", index=False)

In [ ]:
#merged_dents = pd.read_parquet(f"./data-files/insp_{inspection_id}_refined_dents_with_surface.parquet")

In [ ]:
# merged_dents=merged_dents[merged_dents['start_odometer_tick'] != merged_dents['end_odometer_tick'] ]
# display(merged_dents)

## Visualize all detected frames

In [ ]:
# from ilipy.beamformer import BfImageType
# from ilipyutils.beamforming import Beamformer
# FF_DEFAULTS = {
#     "angle_resolution_radians": np.deg2rad(0.025),
#     "radial_resolution_mm": 0.05,
#     "radial_range_mm": [15, 33],
#     "angle_range_radians": np.deg2rad([-11, 11]).tolist(),
# }
# beamformer = Beamformer(
#     session=session,
#     inspection_id=session.active_inspection.inspection_id,
#     clip=clip_ds,
# )
# vol = beamformer.beamform_data(
#     frame_range=(iframe, iframe + 1),
#     bf_type=BfImageType.InnerSurfaceDetect,
#     calc_circles=False,
#     bf_configs=FF_DEFAULTS,
# )

import os
from pathlib import Path
import matplotlib.pyplot as plt
from ilipy.beamformer import BfImageType
from ilipyutils.beamforming import Beamformer
from ilipy import Clip, ClipTypes


def visualize_bookmark_frames(bookmarks_df, session, output_folder="./bookmark_frames"):
    """
    Save beamformed images for all frames in bookmarks dataframe.
    
    Args:
        bookmarks_df: DataFrame with bookmark data containing track_id, start_frame, end_frame
        session: ilipy Session object
        output_folder: Path to save the frame images
    """
    # Create output folder if it doesn't exist
    output_path = Path(output_folder)
    output_path.mkdir(parents=True, exist_ok=True)
    
    # Beamformer configuration
    FF_DEFAULTS = {
        "angle_resolution_radians": np.deg2rad(0.025),
        "radial_resolution_mm": 0.05,
        "radial_range_mm": [15, 33],
        "angle_range_radians": np.deg2rad([-11, 11]).tolist(),
    }
    

    print(f"Processing {len(bookmarks_df)} bookmarks...")
    clips = session.get_clips_by_type(ClipTypes.ChannelData)
    
    # Process each bookmark
    for idx, row in bookmarks_df.iterrows():
        track_id = int(row['track_id'])
        start_frame = int(row['start_frame'])
        end_frame = int(row['end_frame'])
        clip_id = str(row['clip_id'])
        clip = [clip for clip in clips if clip.clip_id==clip_id]
            # Initialize beamformer
        beamformer = Beamformer(
            session=session,
            inspection_id=inspection_id,
            clip=clip[0],
        )
    
        # Create subfolder for each bookmark
        bookmark_folder = output_path / f"bookmark_{idx}_track_{track_id:02d}"
        bookmark_folder.mkdir(exist_ok=True)
        
        print(f"Processing bookmark {idx}: track {track_id}, frames {start_frame}-{end_frame}")
        
        # Process frames in the range
        for frame in range(start_frame, end_frame + 1):
            try:
                # Beamform the frame
                vol = beamformer.beamform_data(
                    frame_range=(frame, frame + 1),
                    bf_type=BfImageType.InnerSurfaceDetect,
                    calc_circles=False,
                    bf_configs=FF_DEFAULTS,
                )
                
                # Create figure
                fig, ax = plt.subplots(figsize=(10, 8))
                
                # Display the beamformed image
                im = ax.imshow(vol[0], cmap='gray', aspect='auto')
                ax.set_title(f"Track {track_id}, Frame {frame}")
                ax.set_xlabel("Angle")
                ax.set_ylabel("Radial Distance")
                plt.colorbar(im, ax=ax, label="Intensity")
                
                # Save the figure
                output_file = bookmark_folder / f"frame_{frame:010d}.png"
                plt.savefig(output_file, dpi=150, bbox_inches='tight')
                plt.close(fig)
                
                print(f"  Saved frame {frame}")
                
            except Exception as e:
                print(f"  Error processing frame {frame}: {e}")
                continue
    
    print(f"\nAll frames saved to {output_folder}")


visualize_bookmark_frames(bookmarks_dfs, session, output_folder="./dent_us_frames_visualization2")

In [3]:
def group_by_view_distance_bins(df, view_distance_start_col='view_distance_start', view_distance_stop_col='view_distance_stop', group_col='view_distance_group', bin_size=0.2):
    """
    Group DataFrame rows based on binned mean of view_distance_start and view_distance_stop
    
    Args:
        df: DataFrame with view distance columns
        view_distance_start_col: column name containing start view distances
        view_distance_stop_col: column name containing stop view distances
        group_col: name for the new grouping column
        bin_size: size of each bin (default 0.2)
    
    Returns:
        DataFrame with added grouping column
    """
    if df is None or len(df) == 0:
        return df
    
    df = df.copy()
    
    # Calculate mean of view_distance_start and view_distance_stop
    if view_distance_stop_col in df.columns:
        view_distance_mean = (df[view_distance_start_col] + df[view_distance_stop_col]) / 2
    else:
        # Fallback to just view_distance_start if view_distance_stop doesn't exist
        view_distance_mean = df[view_distance_start_col]
    
    # Create grouping column based on binned mean view distance
    df[group_col] = (np.floor(view_distance_mean / bin_size) * bin_size).round(1)
    
    return df

def analyze_view_distance_groups(df, view_distance_start_col='view_distance_start', view_distance_stop_col='view_distance_stop', group_col='view_distance_group', bin_size=0.2):
    """
    Group by view distance bins and analyze the groups
    
    Args:
        df: DataFrame with view distances
        view_distance_start_col: column name containing start view distances
        view_distance_stop_col: column name containing stop view distances
        group_col: name for the grouping column
        bin_size: size of each bin (default 0.2)
    
    Returns:
        tuple: (grouped_df, summary_stats)
    """
    # Add grouping column based on mean of start and stop
    grouped_df = group_by_view_distance_bins(df, view_distance_start_col, view_distance_stop_col, group_col, bin_size)
    
    # Create summary statistics with track_id list
    agg_dict = {
        view_distance_start_col: ['count', 'min', 'max', 'mean'],
        'track_id': lambda x: list(x.unique())  # Keep list of unique track IDs
    }
    
    # Add view_distance_stop aggregations if column exists
    if 'view_distance_stop' in df.columns:
        agg_dict['view_distance_stop'] = ['min', 'max', 'mean']
    
    # Add other columns if they exist
    #keep list of max_confidence values

    if 'avg_confidence' in df.columns:
        agg_dict['avg_confidence'] = ['mean', 'max', 'min']
    if 'sequence_length' in df.columns:
        agg_dict['sequence_length'] = ['mean', 'sum']
    
    # Perform the main aggregation without ransac_inlier_ratio_min
    summary_stats = grouped_df.groupby(group_col).agg(agg_dict)
    
    # Flatten column names
    new_columns = []
    for col in summary_stats.columns:
        if col[0] == 'track_id':
            new_columns.append('track_ids')
        else:
            new_columns.append(f'{col[0]}_{col[1]}')
    
    summary_stats.columns = new_columns
    summary_stats = summary_stats.reset_index()
    
    # Add ransac_inlier_ratio_min dictionary separately if column exists
    if 'ransac_inlier_ratio_min' in grouped_df.columns:
        ransac_dict = grouped_df.groupby(group_col).apply(
            lambda g: dict(zip(g['track_id'].values, g['ransac_inlier_ratio_min'].values))
        )
        summary_stats['ransac_inlier_ratio_min_by_track'] = summary_stats[group_col].map(ransac_dict)
    
    # Add max_confidence dictionary by track_id if column exists
    if 'max_confidence' in grouped_df.columns:
        max_conf_dict = grouped_df.groupby(group_col).apply(
            lambda g: dict(zip(g['track_id'].values, g['max_confidence'].values))
        )
        summary_stats['max_confidence_by_track'] = summary_stats[group_col].map(max_conf_dict)
    
    # Add count of unique tracks per group
    summary_stats['unique_tracks_count'] = summary_stats['track_ids'].apply(len)
    
    print(f"Created {len(summary_stats)} view distance groups (bin size: {bin_size})")
    print(f"Number of groups with less than 6 tracks: {len(summary_stats[summary_stats['unique_tracks_count'] <6])}")
    print(f"Group range: {summary_stats[group_col].min()} to {summary_stats[group_col].max()}")
    
    return grouped_df, summary_stats


# Apply to your bookmarks data with 0.2 bin size
bookmarks_dfs = bookmarks_dfs.query('frame_span <= 300 and frame_span >= 5')
grouped_bookmarks, group_summary = analyze_view_distance_groups(bookmarks_dfs, bin_size=0.3)

print("View Distance Groups Summary:")
display(group_summary)

# Example: Show ransac ratios by track for first few groups
print("\nExample ransac_inlier_ratio_min_by_track (first 3 groups):")
for idx, row in group_summary.head(3).iterrows():
    print(f"\nGroup {row['view_distance_group']}:")

Created 29723 view distance groups (bin size: 0.3)
Number of groups with less than 6 tracks: 29553
Group range: 6.9 to 201620.7
View Distance Groups Summary:


/tmp/ipykernel_6357/2832125963.py:90: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  max_conf_dict = grouped_df.groupby(group_col).apply(


,view_distance_group,view_distance_start_count,view_distance_start_min,view_distance_start_max,view_distance_start_mean,track_ids,view_distance_stop_min,view_distance_stop_max,view_distance_stop_mean,avg_confidence_mean,avg_confidence_max,avg_confidence_min,sequence_length_mean,sequence_length_sum,max_confidence_by_track,unique_tracks_count
0,6.9,5,6.861793,7.178679,7.040086,"[16, 18]",7.045522,7.401102,7.176379,0.991960,0.9943,0.9905,76.600000,383,"{16: 0.9958, 18: 0.9912}",2
1,7.2,3,7.239859,7.412087,7.303266,"[16, 18]",7.247056,7.504650,7.355713,0.990733,0.9911,0.9905,30.333333,91,"{16: 0.9922, 18: 0.9913}",2
2,7.5,6,7.570822,7.638140,7.607509,"[5, 7, 15, 16, 18, 19]",7.584421,7.665967,7.635818,0.991450,0.9928,0.9905,17.333333,104,"{5: 0.9938, 7: 0.991, 15: 0.9943, 16: 0.9913, ...",6
3,7.8,3,7.791304,7.860076,7.829494,"[8, 15, 19]",7.855114,8.360874,8.032038,0.993700,0.9951,0.9927,113.333333,340,"{8: 0.9943, 15: 0.9968, 19: 0.9955}",3
4,8.1,2,8.080930,8.151168,8.116049,"[12, 13]",8.151449,8.181559,8.166504,0.993350,0.9941,0.9926,29.000000,58,"{12: 0.9956, 13: 0.994}",2
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
29718,201613.5,1,201613.558576,201613.558576,201613.558576,[14],201613.574779,201613.574779,201613.574779,0.990400,0.9904,0.9904,10.000000,10,{14: 0.9904},1
29719,201613.8,1,201613.620981,201613.620981,201613.620981,[4],201614.078179,201614.078179,201614.078179,0.991800,0.9918,0.9918,255.000000,255,{4: 0.9933},1
29720,201614.7,8,201614.723454,201614.912260,201614.814268,"[11, 12, 13, 17, 18, 19, 20, 21]",201614.775653,201615.101352,201614.951142,0.993838,0.9953,0.9931,75.750000,606,"{11: 0.9957, 12: 0.9956, 13: 0.9944, 17: 0.994...",8
29721,201615.0,2,201615.085357,201615.101602,201615.093480,"[13, 18]",201615.110066,201615.126502,201615.118284,0.991100,0.9912,0.9910,15.000000,30,"{13: 0.9916, 18: 0.9917}",2



Example ransac_inlier_ratio_min_by_track (first 3 groups):

Group 6.9:

Group 7.2:

Group 7.5:


In [ ]:
display(bookmarks_dfs)

In [4]:
group_summary = group_summary[group_summary['unique_tracks_count']<6]
len(group_summary)

29553

In [5]:
#read final_df_a
final_df_ae = pd.read_parquet(f"./insp_{inspection_id}_final_df_ae.parquet")
display(final_df_ae)

,start,end,prob,max_prob,max_prob_track
0,201612.35,201612.65,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0002, 0....",0.9997,10
1,201607.10,201607.40,"[0.0012, 0.0, 0.0015, 0.0004, 0.0, 0.0, 0.0, 0...",0.9999,19
2,201604.85,201605.15,"[0.02, 0.0001, 0.0045, 0.0002, 0.0, 0.0, 0.0, ...",0.9998,19
3,201604.40,201604.70,"[0.001, 0.0, 0.0001, 0.0, 0.0, 0.0001, 0.0, 0....",0.9998,19
4,201594.35,201594.65,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...",0.9997,15
...,...,...,...,...,...
16328,679.70,680.00,"[0.0, 0.9996, 0.9988, 0.0, 0.0, 0.0, 0.0, 0.0,...",0.9996,1
16329,639.05,639.35,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...",0.9999,14
16330,574.40,574.70,"[0.0018, 0.9851, 0.9985, 0.9999, 0.9683, 0.039...",0.9999,3
16331,499.30,499.60,"[0.0, 0.0, 0.0, 0.0059, 0.9998, 0.5611, 0.0, 0...",0.9998,4


In [6]:
def calculate_overlap_percent_with_ae(group_summary, final_df_ae, 
                                      start_mean_col='view_distance_start_mean',
                                      stop_mean_col='view_distance_stop_mean',
                                      ae_start_col='start',
                                      ae_end_col='end',
                                      output_col='overlap_percent_with_ae'):
    """
    Calculate overlap percentage between group_summary view distance ranges 
    and all ranges in final_df_ae (OPTIMIZED VERSION using vectorization).
    
    Args:
        group_summary: DataFrame with view_distance_start_mean and view_distance_stop_mean columns
        final_df_ae: DataFrame with start and end columns (AE ranges)
        start_mean_col: column name in group_summary for start view distance mean
        stop_mean_col: column name in group_summary for stop view distance mean
        ae_start_col: column name in final_df_ae for start
        ae_end_col: column name in final_df_ae for end
        output_col: name for the new overlap percentage column
    
    Returns:
        DataFrame with added overlap_percent_with_ae column
    """
    if group_summary is None or len(group_summary) == 0:
        return group_summary
    
    if final_df_ae is None or len(final_df_ae) == 0:
        group_summary = group_summary.copy()
        group_summary[output_col] = 0.0
        return group_summary
    
    group_summary = group_summary.copy()
    
    # Extract ranges as numpy arrays (much faster than iterrows)
    group_starts = group_summary[start_mean_col].values
    group_stops = group_summary[stop_mean_col].values
    group_lengths = group_stops - group_starts
    
    # Filter out NaN values from final_df_ae
    ae_valid_mask = (
        final_df_ae[ae_start_col].notna() & 
        final_df_ae[ae_end_col].notna()
    )
    ae_starts = final_df_ae.loc[ae_valid_mask, ae_start_col].values
    ae_ends = final_df_ae.loc[ae_valid_mask, ae_end_col].values
    
    # Initialize result array
    overlap_percentages = np.zeros(len(group_summary))
    
    # Vectorized overlap calculation using broadcasting
    # For each group range, calculate overlap with all AE ranges at once
    for i in range(len(group_summary)):
        group_start = group_starts[i]
        group_stop = group_stops[i]
        group_len = group_lengths[i]
        
        # Skip if invalid
        if pd.isna(group_start) or pd.isna(group_stop) or group_len <= 0:
            continue
        
        # Calculate overlaps with all AE ranges using vectorized operations
        # Overlap start = max(group_start, ae_start) for each AE range
        overlap_starts = np.maximum(group_start, ae_starts)
        
        # Overlap end = min(group_stop, ae_end) for each AE range
        overlap_ends = np.minimum(group_stop, ae_ends)
        
        # Calculate overlap lengths (only where overlap_start < overlap_end)
        overlap_lengths = np.maximum(0, overlap_ends - overlap_starts)
        
        # Calculate overlap percentages
        overlap_pcts = (overlap_lengths / group_len) * 100
        
        # Sum all overlaps (cap at 100%)
        total_overlap = np.sum(overlap_pcts)
        overlap_percentages[i] = min(total_overlap, 100.0)
    
    group_summary[output_col] = overlap_percentages
    
    return group_summary

# Apply the optimized function
group_summary_with_overlap = calculate_overlap_percent_with_ae(group_summary, final_df_ae)
#group_summary_with_overlap.to_parquet(f"./insp_{inspection_id}_group_summary_with_overlap.parquet", index=False)
# Display results
display(group_summary_with_overlap)

,view_distance_group,view_distance_start_count,view_distance_start_min,view_distance_start_max,view_distance_start_mean,track_ids,view_distance_stop_min,view_distance_stop_max,view_distance_stop_mean,avg_confidence_mean,avg_confidence_max,avg_confidence_min,sequence_length_mean,sequence_length_sum,max_confidence_by_track,unique_tracks_count,overlap_percent_with_ae
0,6.9,5,6.861793,7.178679,7.040086,"[16, 18]",7.045522,7.401102,7.176379,0.991960,0.9943,0.9905,76.600000,383,"{16: 0.9958, 18: 0.9912}",2,0.0
1,7.2,3,7.239859,7.412087,7.303266,"[16, 18]",7.247056,7.504650,7.355713,0.990733,0.9911,0.9905,30.333333,91,"{16: 0.9922, 18: 0.9913}",2,0.0
3,7.8,3,7.791304,7.860076,7.829494,"[8, 15, 19]",7.855114,8.360874,8.032038,0.993700,0.9951,0.9927,113.333333,340,"{8: 0.9943, 15: 0.9968, 19: 0.9955}",3,0.0
4,8.1,2,8.080930,8.151168,8.116049,"[12, 13]",8.151449,8.181559,8.166504,0.993350,0.9941,0.9926,29.000000,58,"{12: 0.9956, 13: 0.994}",2,0.0
5,8.4,1,8.499537,8.499537,8.499537,[15],8.877546,8.877546,8.877546,0.993400,0.9934,0.9934,211.000000,211,{15: 0.9953},1,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
29717,201613.2,4,201613.098983,201613.360497,201613.258646,"[3, 4, 14]",201613.293806,201613.583181,201613.439120,0.992200,0.9932,0.9912,101.250000,405,"{3: 0.9921, 4: 0.9942, 14: 0.9946}",3,0.0
29718,201613.5,1,201613.558576,201613.558576,201613.558576,[14],201613.574779,201613.574779,201613.574779,0.990400,0.9904,0.9904,10.000000,10,{14: 0.9904},1,0.0
29719,201613.8,1,201613.620981,201613.620981,201613.620981,[4],201614.078179,201614.078179,201614.078179,0.991800,0.9918,0.9918,255.000000,255,{4: 0.9933},1,0.0
29721,201615.0,2,201615.085357,201615.101602,201615.093480,"[13, 18]",201615.110066,201615.126502,201615.118284,0.991100,0.9912,0.9910,15.000000,30,"{13: 0.9916, 18: 0.9917}",2,0.0


In [7]:
print(len(group_summary_with_overlap))
final_df_us = group_summary_with_overlap[group_summary_with_overlap['overlap_percent_with_ae']==0]
print(len(final_df_us))
#final_df_us.to_parquet(f"./insp_{inspection_id}_final_df_us.parquet", index=False)
#make a histogram of the overlap_percent_with_ae

29553
27691


In [8]:
def combine_consecutive_rows_with_equal_tracks(df, track_ids_col='track_ids', 
                                               agg_functions=None,
                                               view_distance_stop_col='view_distance_stop_mean',
                                               view_distance_start_col='view_distance_start_mean',
                                               max_gap=1.0):
    """
    Combine consecutive rows in a DataFrame based on track_ids list comparison:
    - If lists are entirely equal (same track_ids): merge them
    - If lengths are not equal:
      - If either list has length 2: merge if they have 1 common track_id
      - If both lists have length > 2: merge if they have at least 2 common track_ids
    
    Additional condition: Do not merge if view_distance_start_max from first row 
    is more than max_gap smaller than view_distance_start_min in second row.
    
    Args:
        df: DataFrame to process
        track_ids_col: column name containing track_ids list
        agg_functions: dict mapping column names to aggregation functions
                      (e.g., {'view_distance_start_mean': 'mean', 'count': 'sum'})
                      If None, uses default aggregations
        view_distance_stop_col: column name for view distance stop (from first row)
        view_distance_start_col: column name for view distance start (from second row)
        max_gap: maximum allowed gap between rows (default: 1.0)
    
    Returns:
        DataFrame with combined rows
    """
    if df is None or len(df) == 0:
        return df
    
    df = df.copy().reset_index(drop=True)
    
    # Default aggregation functions
    if agg_functions is None:
        agg_functions = {}
    
    # Helper function to check if a value is numeric-compatible
    def is_numeric_type(val):
        """Check if value can be used in numeric operations"""
        return isinstance(val, (int, float, np.integer, np.floating)) and not isinstance(val, bool)
    
    # Helper function to check if column contains non-numeric types
    def column_has_numeric_values(series):
        """Check if series contains numeric values"""
        if len(series) == 0:
            return False
        first_val = series.iloc[0]
        # Check if first value is numeric
        if pd.isna(first_val):
            # Check other values
            for val in series:
                if not pd.isna(val):
                    return is_numeric_type(val)
            return False
        return is_numeric_type(first_val)
    
    # Helper function to merge dictionaries (for max_confidence_by_track, etc.)
    def merge_dicts_max(dict_list):
        """
        Merge multiple dictionaries, taking maximum value for each key.
        Used for max_confidence_by_track and similar dictionary columns.
        """
        if not dict_list:
            return {}
        
        # Filter out None/NaN values
        valid_dicts = [d for d in dict_list if d is not None and isinstance(d, dict)]
        
        if not valid_dicts:
            return {}
        
        # Start with first dictionary
        merged = valid_dicts[0].copy()
        
        # For each subsequent dictionary, take max value for each key
        for d in valid_dicts[1:]:
            for key, value in d.items():
                if key in merged:
                    # Take maximum value
                    merged[key] = max(merged[key], value)
                else:
                    # Add new key
                    merged[key] = value
        
        return merged
    
    # Helper function to check if two track_ids lists should be merged
    def tracks_should_merge(tracks1, tracks2):
        """
        Check if two track_ids lists should be merged based on:
        - If lists are entirely equal (same track_ids): merge them
        - If lengths are not equal:
          - If either list has length 2: merge if they have 1 common track_id
          - If both lists have length > 2: merge if they have at least 2 common track_ids
        """
        # Handle None/NaN cases
        if tracks1 is None and tracks2 is None:
            return True
        if tracks1 is None or tracks2 is None:
            return False
        
        # Check if either is NaN (for scalar values)
        try:
            if pd.isna(tracks1) and pd.isna(tracks2):
                return True
            if pd.isna(tracks1) or pd.isna(tracks2):
                return False
        except (ValueError, TypeError):
            # If pd.isna fails (e.g., for lists), continue with comparison
            pass
        
        # Convert to lists if needed
        if isinstance(tracks1, (list, tuple, np.ndarray)):
            list1 = list(tracks1)
        else:
            list1 = [tracks1] if tracks1 is not None else []
        
        if isinstance(tracks2, (list, tuple, np.ndarray)):
            list2 = list(tracks2)
        else:
            list2 = [tracks2] if tracks2 is not None else []
        
        # Get lengths
        len1 = len(list1)
        len2 = len(list2)
        
        # Handle empty lists
        if len1 == 0 and len2 == 0:
            return True
        if len1 == 0 or len2 == 0:
            return False
        
        # Convert to sets for comparison
        set1 = set(list1)
        set2 = set(list2)
        common_tracks = set1.intersection(set2)
        num_common = len(common_tracks)
        
        # If lists are entirely equal (same track_ids): merge them
        if set1 == set2:
            return True
        
        # If lengths are not equal:
        # - If either list has length 2: merge if they have 1 common track_id
        # - If both lists have length > 2: merge if they have at least 2 common track_ids
        if len1 == 2 or len2 == 2:
            # At least one list has length 2: merge if 1 common track_id
            return num_common >= 1
        else:
            # Both lists have length > 2: merge if at least 2 common track_ids
            return num_common >= 2
    
    def view_distance_gap_ok(row1, row2):
        """Check if view distance gap is acceptable for merging."""
        if view_distance_stop_col not in df.columns or view_distance_start_col not in df.columns:
            return True
        
        vd_stop_1 = row1[view_distance_stop_col]
        vd_start_2 = row2[view_distance_start_col]
        
        if pd.isna(vd_stop_1) or pd.isna(vd_start_2):
            return True
        
        if vd_stop_1 + max_gap < vd_start_2:
            return False
        
        return True
    
    # Identify groups of consecutive rows with common track_ids
    groups = []
    current_group = [0]
    
    for i in range(1, len(df)):
        prev_tracks = df.loc[i-1, track_ids_col]
        curr_tracks = df.loc[i, track_ids_col]
        prev_row = df.loc[i-1]
        curr_row = df.loc[i]
        
        # Check both conditions: track_ids and view distance gap
        tracks_ok = tracks_should_merge(prev_tracks, curr_tracks)
        view_distance_ok = view_distance_gap_ok(prev_row, curr_row)
        
        if tracks_ok and view_distance_ok:
            # Both conditions satisfied - add to current group
            current_group.append(i)
        else:
            # Don't meet merge criteria - save current group and start new one
            groups.append(current_group)
            current_group = [i]
    
    # Add the last group
    groups.append(current_group)
    
    # Combine rows in each group
    combined_rows = []
    
    for group_indices in groups:
        group_df = df.loc[group_indices].copy()
        
        # Create combined row
        combined_row = {}
        
        for col in df.columns:
            if col == track_ids_col:
                # Combine track_ids: union of all track_ids in the group
                all_tracks = []
                for idx in group_indices:
                    tracks = df.loc[idx, track_ids_col]
                    if tracks is not None:
                        if isinstance(tracks, (list, tuple, np.ndarray)):
                            all_tracks.extend(list(tracks))
                        else:
                            all_tracks.append(tracks)
                # Remove duplicates and sort
                combined_row[col] = sorted(list(set(all_tracks)))
            
            elif col in agg_functions:
                # Use specified aggregation function
                agg_func = agg_functions[col]
                
                # Check if column contains non-numeric types (dicts, lists, etc.)
                if not column_has_numeric_values(group_df[col]):
                    # Special handling for dictionary columns
                    first_val = group_df[col].iloc[0]
                    if isinstance(first_val, dict):
                        # Merge dictionaries - collect all dicts and merge them
                        dict_list = group_df[col].tolist()
                        if col in ['max_confidence_by_track', 'ransac_inlier_ratio_min_by_track']:
                            # For max_confidence: take max value for each track_id
                            # For ransac_inlier_ratio_min: take min value for each track_id
                            if col == 'max_confidence_by_track':
                                combined_row[col] = merge_dicts_max(dict_list)
                            elif col == 'ransac_inlier_ratio_min_by_track':
                                # For min ratio, we want minimum values
                                merged = {}
                                valid_dicts = [d for d in dict_list if d is not None and isinstance(d, dict)]
                                if valid_dicts:
                                    merged = valid_dicts[0].copy()
                                    for d in valid_dicts[1:]:
                                        for key, value in d.items():
                                            if key in merged:
                                                merged[key] = min(merged[key], value)
                                            else:
                                                merged[key] = value
                                combined_row[col] = merged
                            else:
                                # Default: merge and take max
                                combined_row[col] = merge_dicts_max(dict_list)
                        else:
                            # For other dict columns, merge with max
                            combined_row[col] = merge_dicts_max(dict_list)
                    else:
                        # For non-numeric, non-dict types, use 'first'
                        combined_row[col] = group_df[col].iloc[0]
                elif agg_func == 'first':
                    combined_row[col] = group_df[col].iloc[0]
                elif agg_func == 'last':
                    combined_row[col] = group_df[col].iloc[-1]
                elif agg_func == 'mean':
                    combined_row[col] = group_df[col].mean()
                elif agg_func == 'sum':
                    combined_row[col] = group_df[col].sum()
                elif agg_func == 'min':
                    combined_row[col] = group_df[col].min()
                elif agg_func == 'max':
                    combined_row[col] = group_df[col].max()
                elif agg_func == 'list':
                    # Combine into list
                    combined_row[col] = group_df[col].tolist()
                else:
                    # Default to first value
                    combined_row[col] = group_df[col].iloc[0]
            
            else:
                # Default aggregation based on column type
                if column_has_numeric_values(group_df[col]):
                    # Numeric: take mean
                    combined_row[col] = group_df[col].mean()
                else:
                    # Non-numeric: check if it's a dictionary
                    first_val = group_df[col].iloc[0]
                    if isinstance(first_val, dict):
                        # Merge dictionaries - collect all dicts and merge them
                        dict_list = group_df[col].tolist()
                        if col == 'max_confidence_by_track':
                            # For max_confidence: take max value for each track_id
                            combined_row[col] = merge_dicts_max(dict_list)
                        elif col == 'ransac_inlier_ratio_min_by_track':
                            # For min ratio: take min value for each track_id
                            merged = {}
                            valid_dicts = [d for d in dict_list if d is not None and isinstance(d, dict)]
                            if valid_dicts:
                                merged = valid_dicts[0].copy()
                                for d in valid_dicts[1:]:
                                    for key, value in d.items():
                                        if key in merged:
                                            merged[key] = min(merged[key], value)
                                        else:
                                            merged[key] = value
                            combined_row[col] = merged
                        else:
                            # Default: merge with max
                            combined_row[col] = merge_dicts_max(dict_list)
                    else:
                        # Non-numeric (lists, strings): take first value
                        combined_row[col] = group_df[col].iloc[0]
        
        combined_rows.append(combined_row)
    
    result_df = pd.DataFrame(combined_rows)
    
    print(f"Combined {len(df)} rows into {len(result_df)} rows")
    
    return result_df

In [9]:
# custom aggregation functions
agg_funcs = {
    'view_distance_start_min': 'min',
    'view_distance_stop_max': 'max',
    'view_distance_start_mean': 'min',
    'view_distance_stop_mean': 'max',
    'view_distance_start_count': 'sum',
    'unique_tracks_count': 'first',  # Keep first value since track_ids are same
    'view_distance_group': 'first',
    'max_confidence_by_track': 'sum',  # Keep first dict
    'ransac_inlier_ratio_min_by_track': 'first'
}

combined_df = combine_consecutive_rows_with_equal_tracks(
    final_df_us, 
    track_ids_col='track_ids',
    agg_functions=agg_funcs,
    max_gap=0.5
)

Combined 27691 rows into 20494 rows


In [10]:
a_old = final_df_us["view_distance_stop_mean"]-final_df_us["view_distance_start_mean"]
a = combined_df["view_distance_stop_mean"]-combined_df["view_distance_start_mean"]
b = combined_df["view_distance_stop_max"]-combined_df["view_distance_start_max"]

In [17]:
#give the top 5 indicies of a with max values
top_5_indices = a.nlargest(5
                           ).index.tolist()
print("Top 5 indices with max (view_distance_stop_mean - view_distance_start_mean):", top_5_indices)
#print combined rows at these indicies
print(a.loc[top_5_indices])
print("Combined rows at top 5 indices:")
display(combined_df.loc[top_5_indices])

Top 5 indices with max (view_distance_stop_mean - view_distance_start_mean): [19321, 16569, 19310, 16633, 17036]
19321    5.893266
16569    3.848933
19310    3.835202
16633    3.697056
17036    3.548905
dtype: float64
Combined rows at top 5 indices:


,view_distance_group,view_distance_start_count,view_distance_start_min,view_distance_start_max,view_distance_start_mean,track_ids,view_distance_stop_min,view_distance_stop_max,view_distance_stop_mean,avg_confidence_mean,avg_confidence_max,avg_confidence_min,sequence_length_mean,sequence_length_sum,max_confidence_by_track,unique_tracks_count,overlap_percent_with_ae
19321,187412.7,31,187412.733363,187415.681505,187412.843206,[15],187415.687884,187418.736472,187418.736472,0.990803,0.990935,0.990671,55.377451,70.411765,{15: 0.9934},1,0.0
16569,175217.7,14,175217.577695,175219.409066,175217.624882,"[3, 4]",175219.651732,175221.498856,175221.473815,0.992772,0.993111,0.992433,148.055556,250.555556,"{3: 0.9956, 4: 0.9963}",2,0.0
19310,187340.7,21,187340.890057,187342.789595,187340.890057,[15],187342.822895,187344.725259,187344.725259,0.990988,0.991142,0.990833,70.000000,100.833333,{15: 0.9935},1,0.0
16633,175579.2,17,175579.186585,175581.261438,175579.186585,[15],175581.354543,175582.928623,175582.883641,0.991821,0.992009,0.991627,98.333333,135.000000,{15: 0.9943},1,0.0
17036,176779.5,17,176779.500074,176781.186776,176779.595494,"[6, 12, 14, 15]",176781.250149,176783.144400,176783.144400,0.992619,0.993122,0.992011,82.870370,140.000000,"{14: 0.9958, 15: 0.9953, 12: 0.9926, 6: 0.9941}",1,0.0


In [ ]:
def add_final_track_id(df, track_ids_col='track_ids', output_col='final_track_id'):
    """
    Add a final_track_id column that contains the track_id from the list 
    that is closest to all other track_ids (minimizes sum of distances).
    
    Args:
        df: DataFrame with track_ids column
        track_ids_col: column name containing list of track_ids
        output_col: name for the new final_track_id column
    
    Returns:
        DataFrame with added final_track_id column
    """
    if df is None or len(df) == 0:
        return df
    
    df = df.copy()
    final_track_ids = []
    
    def find_closest_track(track_list):
        """
        Find the track_id that minimizes the sum of absolute differences 
        to all other track_ids in the list.
        """
        if track_list is None or len(track_list) == 0:
            return None
        
        # Convert to list if needed
        if isinstance(track_list, (list, tuple, np.ndarray)):
            tracks = list(track_list)
        else:
            tracks = [track_list]
        
        # Remove any None/NaN values
        tracks = [t for t in tracks if t is not None and not pd.isna(t)]
        
        if len(tracks) == 0:
            return None
        
        # If only one track, return it
        if len(tracks) == 1:
            return tracks[0]
        
        # Convert to numeric (in case they're strings)
        try:
            tracks = [int(t) for t in tracks]
        except (ValueError, TypeError):
            # If conversion fails, use as-is
            pass
        
        # Find track_id that minimizes sum of absolute differences to all others
        min_total_distance = float('inf')
        closest_track = tracks[0]
        
        for candidate_track in tracks:
            # Calculate sum of absolute differences to all other tracks
            total_distance = sum(abs(candidate_track - other_track) for other_track in tracks)
            
            if total_distance < min_total_distance:
                min_total_distance = total_distance
                closest_track = candidate_track
        
        return closest_track
    
    # Process each row
    for idx, row in df.iterrows():
        track_list = row[track_ids_col]
        final_track = find_closest_track(track_list)
        final_track_ids.append(final_track)
    
    df[output_col] = final_track_ids
    
    return df

In [ ]:
df_with_final = add_final_track_id(combined_df, track_ids_col='track_ids', output_col='final_track_id')

In [ ]:
# Convert dict keys to strings for parquet compatibility
def convert_dict_keys_to_str(d):
    """Convert dictionary keys to strings for parquet compatibility."""
    if d is None or not isinstance(d, dict):
        return d
    return {str(k): v for k, v in d.items()}

# Apply to columns with dict values
df_to_save = df_with_final.copy()
if 'max_confidence_by_track' in df_to_save.columns:
    df_to_save['max_confidence_by_track'] = df_to_save['max_confidence_by_track'].apply(convert_dict_keys_to_str)
if 'ransac_inlier_ratio_min_by_track' in df_to_save.columns:
    df_to_save['ransac_inlier_ratio_min_by_track'] = df_to_save['ransac_inlier_ratio_min_by_track'].apply(convert_dict_keys_to_str)

# Save to parquet
df_to_save.to_parquet(f"./insp_{inspection_id}_final_df_us_with_final_track_id.parquet", index=False)

In [ ]:

df_with_final_track = pd.read_parquet(f"./insp_{inspection_id}_final_df_us_with_final_track_id.parquet")
display(df_with_final_track)

## Upload anomalies

In [ ]:
from ilipy import ClipTypes, Session, TrackIndex, ViewDistance
from ilipy.channeldata import ImageProfile
from ilipy.features import Bookmarks
from ilipyutils.ml_features.base import (get_ml_models_info_list, AnomalyStatus)
from ilipyutils.ml_features.insert import FeatureInsert, GeometryCubeParameters
from ili_custom_data.load_model import ModelDataLoader
from ilipy.database import DistanceCorrelation
from ilipyutils.ml_features.overlap import (
    AnomalyOverlapManager,
    OverlapAction,
    OverlapPolicy,
)


session = Session(environment=environment)
session.set_active_inspection(inspection_id)
dist_corr = DistanceCorrelation(session)
bookmarks_interface = Bookmarks(connector=session.database_connector)
latest_model = ModelDataLoader.get_latest_model()

overlap_policy = OverlapPolicy(
    action=OverlapAction.SKIP,
    min_overlap_iou=0.1,
)
anomaly_overlap_manager = AnomalyOverlapManager(
    overlap_policy=overlap_policy,
    session=session,
    bookmarks_interface=bookmarks_interface,
)
feature_insert = FeatureInsert(
    session=session,
    bookmarks_interface=bookmarks_interface,
    anomaly_overlap_manager=anomaly_overlap_manager, # Set to None to disable overlap checking
)

In [ ]:
cube_params = GeometryCubeParameters(
tlbr_cube_angle_rad=(np.deg2rad(-15), np.deg2rad(15)),
tlbr_probe_scan_angle_rad=(np.deg2rad(-15), np.deg2rad(15)),
tlbr_radial_position_mm=(200-(1.9/2), 200+(1.9/2)),
)
model_info = next(
    model
    for model in get_ml_models_info_list()
    if model.model_name == "Ultrasound-Dent-v1"
)
model_info


In [ ]:

#reset index of final_grouped_bookmarks
final_grouped_bookmarks = df_with_final.reset_index(drop=True)
display(final_grouped_bookmarks)

In [ ]:
anomalies = bookmarks_interface.get_anomalies(inspectionId=inspection_id)
dent_anomalies = [a for a in anomalies if "Ultrasound-Dent-v1" in a.tags]
print(f"{len(dent_anomalies)} anomalies with Ultrasound-Dent-v1 tag found in {environment}.")


In [ ]:
# 
# deleted_anomalies = [bookmarks_interface.delete_anomaly(a.feature.anomaly_feature_id) for a in dent_anomalies]

In [ ]:

for i, row in final_grouped_bookmarks.iterrows():
       
    profile = ImageProfile.ZeroAngle 
    vd_start, vd_end, track_id = row['view_distance_start_mean'], row['view_distance_stop_mean'], row['final_track_id']

        
    track_nums = (track_id,track_id)
    start_frame_odo_ticks = dist_corr.get_odometer_ticks_from_view_distance(TrackIndex(track_nums[0]), ViewDistance(vd_start)).value
    end_frame_odo_ticks = dist_corr.get_odometer_ticks_from_view_distance(TrackIndex(track_nums[1]), ViewDistance(vd_end)).value

    print(f"uploading row {i+1} of {len(final_grouped_bookmarks)}")
    anomaly_info = feature_insert.insert_ml_pred(
    inspection_id=inspection_id,
    tlbr_track_indices=track_nums,
    tlbr_odometer_ticks=(start_frame_odo_ticks, end_frame_odo_ticks),
    cube_params=cube_params,
    model_info=model_info,
    anomaly_status=AnomalyStatus.REVIEW_DETECTION,
    image_profile=profile,
    extra_tags=None,
    # ili_custom_data=model_instance
    )
                
